In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from google import genai
import os

from dotenv import load_dotenv
load_dotenv() 

API_key = os.environ["GOOGLE_API_KEY"]
client = genai.Client(api_key=API_key)

In [3]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Write a 4 line poem for Real Madrid"
)

print(response.text)

In spotless white beneath the light,
The kings of Europe claim the night.
With golden crowns and history's flame,
"Hala Madrid!"—they rule the game.


In [4]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path='q_a_db.csv', source_column="prompt")

# Store the loaded data in the 'data' variable
data = loader.load()

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

instructor_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=API_key
)

In [6]:
e = instructor_embeddings.embed_query("How can I change menu prices")

print(len(e))

3072


In [7]:
from langchain_community.vectorstores import FAISS

# Create a FAISS instance for vector database from 'data'
vectordb = FAISS.from_documents(documents=data,
                                 embedding=instructor_embeddings)

# Create a retriever for querying the vector database
retriever = vectordb.as_retriever(score_threshold = 0.7)

In [8]:
rdocs = retriever.invoke("How can I change menu prices")

rdocs

[Document(id='80eaa313-6def-4a2d-a85b-d2a327b84f5d', metadata={'source': 'I want to increase the menu prices by 10%.', 'row': 1}, page_content='prompt: I want to increase the menu prices by 10%.\nresponse: You can apply the changes by the app that is installed on your phone. You can have access to the "Edit Prices" on the "Edit Menu" section in the app. In case we will report the issue to the "Menu Team" to call you for further details.'),
 Document(id='d2278f71-ef81-435a-a4c7-cc7e116fadfb', metadata={'source': 'Hi I want to change the name of one product name.', 'row': 2}, page_content='prompt: Hi I want to change the name of one product name.\nresponse: You can change the name of product by using the installed app on your phone. There is a "Menu Products Change" button on "Edit Menu" section. In case we will report the issue to the "Menu Team" to call you for further details.'),
 Document(id='317c9885-2078-4e5b-ad19-ce1dff78300c', metadata={'source': 'I want to remove one product fro

In [9]:
from langchain_core.prompts import PromptTemplate

prompt_template = """Given the following context and a question, generate an answer based on this context only.
In the answer try to provide as much text as possible from "response" section in the source document context without making much changes.
If the answer is not found in the context, kindly state "I don't have enough informations regarding this matter. In case, I will report it to the Support team to call you for further details." Don't try to make up an answer.

CONTEXT: {context}

QUESTION: {question}"""


PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)
chain_type_kwargs = {"prompt": PROMPT}

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=API_key,
    temperature=0.7
)

In [11]:
from langchain_classic.chains import RetrievalQA

chain = RetrievalQA.from_chain_type(llm=llm,
                            chain_type="stuff",
                            retriever=retriever,
                            input_key="query",
                            return_source_documents=True,
                            chain_type_kwargs=chain_type_kwargs)


In [12]:
chain('How can I change menu prices')

C:\Users\SPINO SHOP\AppData\Local\Temp\ipykernel_19864\1254068145.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  chain('How can I change menu prices')


{'query': 'How can I change menu prices',
 'result': 'You can apply the changes by the app that is installed on your phone. You can have access to the "Edit Prices" on the "Edit Menu" section in the app. In case we will report the issue to the "Menu Team" to call you for further details.',
 'source_documents': [Document(id='80eaa313-6def-4a2d-a85b-d2a327b84f5d', metadata={'source': 'I want to increase the menu prices by 10%.', 'row': 1}, page_content='prompt: I want to increase the menu prices by 10%.\nresponse: You can apply the changes by the app that is installed on your phone. You can have access to the "Edit Prices" on the "Edit Menu" section in the app. In case we will report the issue to the "Menu Team" to call you for further details.'),
  Document(id='d2278f71-ef81-435a-a4c7-cc7e116fadfb', metadata={'source': 'Hi I want to change the name of one product name.', 'row': 2}, page_content='prompt: Hi I want to change the name of one product name.\nresponse: You can change the name

In [13]:
chain('I want to speak with onboarding team')

{'query': 'I want to speak with onboarding team',
 'result': 'You can reach the "Onboarding Team" by calling the number 0145***91',
 'source_documents': [Document(id='20fb64de-b82a-4abb-a4c0-6120fddad2ae', metadata={'source': 'I want to speak with Onboarding team. How can I connect to them?', 'row': 46}, page_content='prompt: I want to speak with Onboarding team. How can I connect to them?\nresponse: You can reach the "Onboarding Team" by calling the number 0145***91'),
  Document(id='6fdf8f4d-38d3-4b10-a00d-27cf3276b760', metadata={'source': 'I want to speak with Support team. How can I connect to them?', 'row': 4}, page_content='prompt: I want to speak with Support team. How can I connect to them?\nresponse: You can reach the "Support Team" by calling the number 0145***95'),
  Document(id='23395bb4-9606-43b4-91ea-85c6b07c1f0b', metadata={'source': 'I want to speak with Partner Care team. How can I connect to them?', 'row': 48}, page_content='prompt: I want to speak with Partner Care 

In [14]:
chain('I want to speak with Partner care team')

{'query': 'I want to speak with Partner care team',
 'result': 'You can reach the "Partner CareTeam" by calling the number 0145***92',
 'source_documents': [Document(id='23395bb4-9606-43b4-91ea-85c6b07c1f0b', metadata={'source': 'I want to speak with Partner Care team. How can I connect to them?', 'row': 48}, page_content='prompt: I want to speak with Partner Care team. How can I connect to them?\nresponse: You can reach the "Partner CareTeam" by calling the number 0145***92'),
  Document(id='582d6d30-6823-4da7-bf36-e7ab974df30c', metadata={'source': 'I want to speak with Care team. How can I connect to them?', 'row': 45}, page_content='prompt: I want to speak with Care team. How can I connect to them?\nresponse: You can reach the "Care Team" by calling the number 0145***94'),
  Document(id='32682cc0-b814-4450-ad5b-424a4eee7c5b', metadata={'source': 'I want to cancell my deal', 'row': 20}, page_content='prompt: I want to cancell my deal\nresponse: I am sorry for the inconvenience. I wi

In [15]:
chain('how long it takes to get the till')

{'query': 'how long it takes to get the till',
 'result': "I don't have enough informations regarding this matter. In case, I will report it to the Support team to call you for further details.",
 'source_documents': [Document(id='489ebac5-5961-411c-9d45-b3e50ecc71bb', metadata={'source': 'How much is your rent for the till?', 'row': 15}, page_content='prompt: How much is your rent for the till?\nresponse: 500£ at first of contract and 13.20£ weekly'),
  Document(id='33ad0f92-9d29-4d82-a163-2ea5573c5f71', metadata={'source': 'I want to change the deliver and collection time on till and website', 'row': 8}, page_content='prompt: I want to change the deliver and collection time on till and website\nresponse: Thanks for the report. On till on the Market app, there is a section named: "Delivery and Collection Time" on the system setup setting. You can change the time over there and "Sync it with the website" to apply it on your website.'),
  Document(id='b7df18e9-0dae-45e6-8db5-4e48677888c

# Thanks for attention
## Mohammadreza